In [1]:
from astropy.table import Table
import numpy as np
import matplotlib.pyplot as plt
import glob
import os
import fitsio
import healpy
from scipy.spatial import cKDTree
from astropy.cosmology import Planck18 as cosmo
from mpl_toolkits.mplot3d import Axes3D
import astropy.units as u
import astropy.constants as const
import matplotlib
import astropy.io.ascii

In [2]:
cylh = 20*u.Mpc
cyld = 2*u.Mpc  #radius is 1 Mpc
rsphere = np.sqrt((cylh/2)**2+(cyld/2)**2) #radius of the sphere of our initial search

#define pruning algorithm
hsq = cylh**2/4
dsq = cyld**2/4

def raDecToUnitSphere(a, d):
    a = a*np.pi/180
    d = d*np.pi/180
    x = np.cos(a)*np.cos(d)
    y = np.sin(a)*np.cos(d)
    z = np.sin(d)
    return x,y,z

def raDecToCartesian(a, d, r):
    return r*raDecToUnitSphere(a,d)*u.Mpc

def checkZBoundary(table,zmin,zmax):
    return (table['d'] > cosmo.comoving_distance(zmin)+cylh/2)&(table['d'] < cosmo.comoving_distance(zmax)-cylh/2)

def checkcylinder(x1,y1,z1,d1,x2,y2,z2,d2):
    #check if two points ar in cylinder with axis along line of sight
    dDsq = (d1 - d2)**2*u.Mpc**2
    if dDsq > hsq: return False
    elif ((x1 - x2)**2 + (y1 - y2)**2 + (z1 - z2)**2)*u.Mpc**2 - dDsq > dsq: 
        return False
    else: return True

#try to optimize by plugging in whole neighbor list at once
#also try to put x,y,z,d all in one column as a list

def tableprune(table1, table2, i, j):
    #i is the index of the primary galaxy (in table 1)
    #j is the index of the secondary galaxy (in table 2)
    if (i % 100000 == 0): print(i)
    return checkcylinder(table1[i]['x'],table1[i]['y'],table1[i]['z'],table1[i]['d'],
                         table2[j]['x'],table2[j]['y'],table2[j]['z'],table2[j]['d'])

In [3]:
mocknumber = 0
mockdir = "/global/cfs/cdirs/desi//survey/catalogs/Y1/mocks/SecondGenMocks/AbacusSummit_v4_2/"
elgfile = mockdir+"mock"+str(mocknumber)+"/ELG_LOP_complete_clustering.dat.fits"
lrgfile = mockdir+"mock"+str(mocknumber)+"/LRG_complete_clustering.dat.fits"
columns = ['RA','DEC','Z','ZWARN']
spelg = Table(fitsio.read(elgfile,columns=columns))
splrg = Table(fitsio.read(lrgfile,columns=columns))

In [4]:
elgsecondaries = spelg[(spelg["ZWARN"]==0)&(spelg["Z"] > 0.75)&(spelg["Z"] < 1)]
lrgsecondaries = splrg[(splrg["ZWARN"]==0)&(splrg["Z"] > 0.75)&(splrg["Z"] < 1)]

In [5]:
#get comoving distances of galaxies along line of sight
elgsecondaries['d'] = cosmo.comoving_distance(elgsecondaries['Z'])
lrgsecondaries['d'] = cosmo.comoving_distance(lrgsecondaries['Z'])

#get Cartesian coordinates of galaxies
elgsecondaries['x'],elgsecondaries['y'],elgsecondaries['z'] = raDecToCartesian(elgsecondaries['RA'],elgsecondaries['DEC'],elgsecondaries['d'])
lrgsecondaries['x'],lrgsecondaries['y'],lrgsecondaries['z'] = raDecToCartesian(lrgsecondaries['RA'],lrgsecondaries['DEC'],lrgsecondaries['d'])

In [6]:
zmin = 0.75
zmax = 1
elgprimaries = elgsecondaries[checkZBoundary(elgsecondaries,zmin,zmax)]
lrgprimaries = lrgsecondaries[checkZBoundary(lrgsecondaries,zmin,zmax)]

In [7]:
#make 3D maps of galaxies
#primaries
elgprimarymap = np.transpose([elgprimaries['x'],elgprimaries['y'],elgprimaries['z']])
lrgprimarymap = np.transpose([lrgprimaries['x'],lrgprimaries['y'],lrgprimaries['z']])
#secondaries
elgsecondarymap = np.transpose([elgsecondaries['x'],elgsecondaries['y'],elgsecondaries['z']])
lrgsecondarymap = np.transpose([lrgsecondaries['x'],lrgsecondaries['y'],lrgsecondaries['z']])
#make ckdtrees
elgtree = cKDTree(elgsecondarymap)
lrgtree = cKDTree(lrgsecondarymap)

In [8]:
#search for neighbors
elgprimaries['neighbors'] = elgtree.query_ball_point(elgprimarymap, rsphere)
lrgprimaries['neighbors'] = lrgtree.query_ball_point(lrgprimarymap, rsphere)

#search for neighbors among other class of galaxies
elgprimaries['lrg_neighbors'] = lrgtree.query_ball_point(elgprimarymap, rsphere)
lrgprimaries['elg_neighbors'] = elgtree.query_ball_point(lrgprimarymap, rsphere)

In [9]:
#prune trees:
elgprimaries['neighborspruned'] = [np.array(elgprimaries[i]['neighbors'])[[tableprune(elgprimaries,elgsecondaries,i,j)
            for j in elgprimaries[i]['neighbors']]][1:] for i in range(len(elgprimaries))]
lrgprimaries['neighborspruned'] = [np.array(lrgprimaries[i]['neighbors'])[[tableprune(lrgprimaries,lrgsecondaries,i,j)
            for j in lrgprimaries[i]['neighbors']]][1:] for i in range(len(lrgprimaries))]
elgprimaries['lrg_neighborspruned'] = [np.array(elgprimaries[i]['lrg_neighbors'])[[tableprune(elgprimaries,lrgsecondaries,i,j)
            for j in elgprimaries[i]['lrg_neighbors']]] for i in range(len(elgprimaries))]
lrgprimaries['elg_neighborspruned'] = [np.array(lrgprimaries[i]['elg_neighbors'])[[tableprune(lrgprimaries,elgsecondaries,i,j) 
            for j in lrgprimaries[i]['elg_neighbors']]] for i in range(len(lrgprimaries))]

0
100000
100000
100000
100000
100000
200000
200000
300000
400000
500000
500000
500000
600000
600000
600000
600000
700000
700000
800000
800000
800000
800000
900000
1000000
1000000
1100000
1100000
1200000
1300000
1300000
1400000
1400000
1400000
1400000
1400000
1400000
1500000
1500000
1600000
1600000
1700000
1700000
1700000
1700000
1800000
1800000
1800000
1900000
2000000
2100000
2100000
2100000
2100000
2200000
2200000
2300000
2300000
2400000
2400000
2400000
2400000
2500000
2500000
2500000
2600000
2700000
2700000
2700000
2800000
2800000
2800000
2900000
3000000
3000000
3000000
3000000
3100000
3100000
3100000
3100000
0
0
0
100000
200000
200000
300000
400000
400000
400000
400000
500000
500000
500000
500000
500000
500000
500000
500000
500000
500000
600000
600000
600000
600000
700000
700000
700000
700000
700000
800000
800000
800000
900000
900000
900000
1000000
1000000
1000000
1000000
1000000
1000000
1100000
1100000
1200000
1200000
1300000
1300000
1300000
1300000
1300000
1300000
1400000
1400000


In [10]:
#count up the galaxies in cylinders
elgprimaries['N_CiC'] = [len(n) for n in elgprimaries['neighborspruned']]
lrgprimaries['N_CiC'] = [len(n) for n in lrgprimaries['neighborspruned']]
elgprimaries['N_lrgCiC'] = [len(n) for n in elgprimaries['lrg_neighborspruned']]
lrgprimaries['N_elgCiC'] = [len(n) for n in lrgprimaries['elg_neighborspruned']]

In [11]:
elgtowrite = elgprimaries['N_CiC','N_lrgCiC','x','y','z','Z','RA','DEC']
lrgtowrite = lrgprimaries['N_CiC','N_elgCiC','x','y','z','Z','RA','DEC']

astropy.io.ascii.write(elgtowrite, 'datafiles/mock0_elg.csv', overwrite=True, format='csv')
astropy.io.ascii.write(lrgtowrite, 'datafiles/mock0_lrg.csv', overwrite=True, format='csv')

In [12]:
elgprimaries[1]

RA,DEC,Z,ZWARN,d,x,y,z,neighbors,lrg_neighbors,neighborspruned,lrg_neighborspruned,N_CiC,N_lrgCiC
,,,,Mpc,Mpc,Mpc,Mpc,,,,,,
float32,float32,float32,int64,float64,float64,float64,float64,object,object,object,object,int64,int64
69.94936,-2.0463943,0.96644473,0,3311.4134297685055,1134.5955011416675,3108.724851611616,-118.24635302307028,[1],"[450483, 1153394, 1220297, 1391907, 1526755]",[],[],0,0


In [13]:
lrgsecondaries[1]

RA,DEC,Z,ZWARN,d,x,y,z
,,,,Mpc,Mpc,Mpc,Mpc
float32,float32,float32,int64,float64,float64,float64,float64
169.87204,-3.5408707,0.8570048,0,3024.7805151330717,-2971.962473110284,530.8837480767298,-186.8120248361076


In [111]:
print(file)

/global/cfs/cdirs/desi//survey/catalogs/Y1/mocks/SecondGenMocks/AbacusSummit_v4_2/mock0/ELG_LOP_complete_clustering.dat.fits
